# Preprocessing OPD — Daftar Nama & Alamat OPD Kab Batang

Notebook ini parse `data/pdf/Nama dan Alamat OPD Kab Batang.pdf` (3 halaman, tabular directory) menjadi list of LangChain `Document` yang siap di-embed.

**Output**: `data/cleaned_opd_docs.pkl` — dikonsumsi `build_vectorstore_opd.ipynb`.

**Pendekatan**: 1 row tabel = 1 Document (atomic chunk), karena nature data directory (lookup, bukan narrative).

**Tools**: `pdfplumber` (table-aware extraction) — sudah ditambahkan ke venv via `uv add pdfplumber`.

## Step 1 — Extract Tables dari PDF

`pdfplumber.extract_tables()` mengembalikan list-of-list of cell strings. Setiap halaman ada 1 tabel.

Catatan dari hasil probe:
- Bullet character `\uf0b7` muncul di kolom alamat/email — perlu di-strip
- Sub-entry: `nomor` cell-nya `None`/`''` dan `nama` mulai dengan pattern `[a-z]. `
- Continuation row (cross-page): `nomor` kosong dan `nama` BUKAN pattern sub-entry → kelanjutan baris sebelumnya
- Footer halaman 3 (tanda tangan Bupati) **tidak masuk** ke tabel → no extra handling

In [1]:
import pdfplumber
from pathlib import Path

PDF_PATH = Path("../data/pdf/Nama dan Alamat OPD Kab Batang.pdf")

raw_rows = []
with pdfplumber.open(PDF_PATH) as pdf:
    for page_idx, page in enumerate(pdf.pages):
        for table in page.extract_tables():
            for row in table:
                # Skip header row (page 1 row 0)
                if row and row[0] and str(row[0]).strip().upper() == "NO":
                    continue
                raw_rows.append({"page": page_idx + 1, "cells": row})

print(f"Total raw rows extracted: {len(raw_rows)}")
print(f"\nFirst 3 rows:")
for r in raw_rows[:3]:
    print(f"  page={r['page']} cells={r['cells']}")

Total raw rows extracted: 62

First 3 rows:
  page=1 cells=['1.', 'Sekretariat Daerah', '\uf0b7 Jl. RA Kartini No. 1 Batang', '(0285) 391571']
  page=1 cells=[None, 'a. Bagian Pemerintahan', '\uf0b7 Jl. RA Kartini No. 1 Batang\n\uf0b7 bag_pemerintahan@batangkab.go.id', '(0285) 392729\n/\n(0285) 391571']
  page=1 cells=[None, 'b. Bagian Kesejahteraan Rakyat', '\uf0b7 Jl. RA Kartini No. 1 Batang\n\uf0b7 bag_kesra@batangkab.go.id', '(0285) 391412\n/\n(0285) 391571']


### Step 1b — Merge Continuation Rows (Cross-Page)

Baris ke-12 (Dinas Pariwisata, Kepemudaan dan...) terpotong di halaman 1, lanjut di halaman 2 row 0 (`Olahraga` + email). Detect: row dengan `nomor` kosong dan `nama` TIDAK match pattern sub-entry → append ke baris sebelumnya.

In [2]:
import re

SUB_ENTRY_RE = re.compile(r"^[a-z]\.\s")
BULLET = "\uf0b7"


def is_empty(v):
    return v is None or str(v).strip() == ""


def is_sub_entry(nama_cell):
    return bool(nama_cell) and bool(SUB_ENTRY_RE.match(nama_cell.strip()))


def merge_cell(a, b, sep=" "):
    a = (a or "").strip()
    b = (b or "").strip()
    if not a:
        return b
    if not b:
        return a
    return f"{a}{sep}{b}"


merged_rows = []
for r in raw_rows:
    cells = r["cells"]
    nomor, nama, alamat_email, telp = cells[0], cells[1], cells[2], cells[3]

    # Continuation row: nomor kosong DAN nama tidak match sub-entry pattern
    if is_empty(nomor) and not is_sub_entry(nama or ""):
        if merged_rows:
            prev = merged_rows[-1]
            prev["nama"] = merge_cell(prev["nama"], nama)
            prev["alamat_email"] = merge_cell(prev["alamat_email"], alamat_email, sep="\n")
            prev["telp"] = merge_cell(prev["telp"], telp, sep="\n")
        continue

    merged_rows.append({
        "page": r["page"],
        "nomor": nomor,
        "nama": nama,
        "alamat_email": alamat_email,
        "telp": telp,
    })

print(f"After merging continuation: {len(merged_rows)} rows (dari {len(raw_rows)})")

# Verify entry #12 sekarang lengkap
for r in merged_rows:
    if r["nomor"] and "12" in r["nomor"]:
        print(f"\nEntry #12 setelah merge:")
        print(f"  nama: {r['nama']!r}")
        print(f"  alamat_email: {r['alamat_email']!r}")
        print(f"  telp: {r['telp']!r}")
        break

After merging continuation: 61 rows (dari 62)

Entry #12 setelah merge:
  nama: 'Dinas Pariwisata, Kepemudaan dan Olahraga'
  alamat_email: '\uf0b7 Jl. Urip Sumoharjo No. 34 Batang\n\uf0b7 disparpora@batangkab.go.id'
  telp: '(0285) 391141'


## Step 2 — Parse Rows → Structured Records

Schema:
```
{
  "nomor":      "1" | "1.a" | "25.b",
  "nama_opd":   "Sekretariat Daerah",
  "parent_opd": None | "Sekretariat Daerah",
  "tipe":       "Sekretariat" | "Bagian" | ...,
  "alamat":     "Jl. RA Kartini No. 1 Batang",
  "email":      "bag_pemerintahan@batangkab.go.id" | None,
  "no_telp":    ["(0285) 392729", "(0285) 391571"],
  "page":       1,
}
```

In [3]:
import re

TIPE_KEYWORDS = [
    "Sekretariat", "Bagian", "Inspektorat", "Badan", "Dinas",
    "Satpol", "Kantor", "Kecamatan", "Kelurahan", "RSUD",
]

EMAIL_RE = re.compile(r"[\w.-]+@[\w.-]+")


def clean_text(s):
    if s is None:
        return ""
    s = s.replace(BULLET, "")
    s = re.sub(r"[ \t]+", " ", s)
    return s.strip()


def parse_alamat_email(cell):
    """Return (alamat_str, email_or_None).
    Lines yang mengandung '@' = email; lines lain = alamat (di-join dengan spasi).
    """
    if not cell:
        return "", None
    lines = [clean_text(ln) for ln in cell.split("\n")]
    lines = [ln for ln in lines if ln]
    alamat_parts, email = [], None
    for ln in lines:
        m = EMAIL_RE.search(ln)
        if m:
            email = m.group(0).strip().rstrip(".")
        else:
            alamat_parts.append(ln)
    alamat = " ".join(alamat_parts).strip()
    return alamat, email


def parse_telp(cell):
    """Return list of normalized phone strings, e.g. ['(0285) 392729', '(0285) 391571']."""
    if not cell:
        return []
    text = clean_text(cell.replace("\n", " "))
    parts = [p.strip() for p in text.split("/")]
    return [p for p in parts if p]


def infer_tipe(nama):
    nama = nama.strip()
    for kw in TIPE_KEYWORDS:
        if nama.startswith(kw) or nama.startswith(f"{kw} "):
            return kw
    return "Lainnya"


def parse_nama_and_subid(nama_raw):
    """Untuk sub-entry 'a. Bagian Pemerintahan' → ('a', 'Bagian Pemerintahan').
    Untuk main entry 'Sekretariat Daerah' → (None, 'Sekretariat Daerah').
    """
    nama_raw = clean_text(nama_raw)
    m = re.match(r"^([a-z])\.\s+(.+)$", nama_raw)
    if m:
        return m.group(1), m.group(2).strip()
    return None, nama_raw


def parse_main_nomor(nomor_raw):
    """'1.' → '1'."""
    nomor_raw = clean_text(nomor_raw)
    return nomor_raw.rstrip(".")

In [4]:
records = []
current_main_nomor = None
current_main_nama = None

for r in merged_rows:
    sub_letter, nama_clean = parse_nama_and_subid(r["nama"])
    alamat, email = parse_alamat_email(r["alamat_email"])
    no_telp = parse_telp(r["telp"])

    if sub_letter is None:
        # Main entry
        nomor = parse_main_nomor(r["nomor"])
        current_main_nomor = nomor
        current_main_nama = nama_clean
        parent = None
    else:
        # Sub-entry — gunakan parent dari main terakhir
        nomor = f"{current_main_nomor}.{sub_letter}"
        parent = current_main_nama

    records.append({
        "nomor": nomor,
        "nama_opd": nama_clean,
        "parent_opd": parent,
        "tipe": infer_tipe(nama_clean),
        "alamat": alamat,
        "email": email,
        "no_telp": no_telp,
        "page": r["page"],
    })

print(f"Total records: {len(records)}")
print(f"\nFirst 3 records:")
for rec in records[:3]:
    print(f"  {rec}")

Total records: 61

First 3 records:
  {'nomor': '1', 'nama_opd': 'Sekretariat Daerah', 'parent_opd': None, 'tipe': 'Sekretariat', 'alamat': 'Jl. RA Kartini No. 1 Batang', 'email': None, 'no_telp': ['(0285) 391571'], 'page': 1}
  {'nomor': '1.a', 'nama_opd': 'Bagian Pemerintahan', 'parent_opd': 'Sekretariat Daerah', 'tipe': 'Bagian', 'alamat': 'Jl. RA Kartini No. 1 Batang', 'email': 'bag_pemerintahan@batangkab.go.id', 'no_telp': ['(0285) 392729', '(0285) 391571'], 'page': 1}
  {'nomor': '1.b', 'nama_opd': 'Bagian Kesejahteraan Rakyat', 'parent_opd': 'Sekretariat Daerah', 'tipe': 'Bagian', 'alamat': 'Jl. RA Kartini No. 1 Batang', 'email': 'bag_kesra@batangkab.go.id', 'no_telp': ['(0285) 391412', '(0285) 391571'], 'page': 1}


## Step 3 — Quality Validation

Automated checks:
- Total records masuk akal (60–65)
- Nomor 1–43 semua ada (no missing main entry)
- Distribusi tipe
- Count missing email / telp (sanity only — memang ada yang kosong di sumber)
- Flag record dengan field aneh

In [5]:
from collections import Counter

# 1. Total count sanity
assert 55 <= len(records) <= 70, f"Unexpected total: {len(records)}"
print(f"✓ Total records: {len(records)} (expected 60-65)")

# 2. Nomor 1-43 semua ada sebagai main entry
main_nomors = {r["nomor"] for r in records if "." not in r["nomor"]}
missing = sorted(set(str(i) for i in range(1, 44)) - main_nomors, key=int)
assert not missing, f"Missing main entries: {missing}"
print(f"✓ Main entries 1-43 lengkap")

# 3. Distribusi tipe
tipe_counts = Counter(r["tipe"] for r in records)
print(f"\nDistribusi tipe:")
for t, n in tipe_counts.most_common():
    print(f"  {t:15s}: {n}")
assert tipe_counts.get("Lainnya", 0) == 0, \
    f"Ada {tipe_counts['Lainnya']} record dengan tipe 'Lainnya' — review nama OPD-nya"

# 4. Field kosong (sanity, bukan error)
no_email = [r for r in records if not r["email"]]
no_telp = [r for r in records if not r["no_telp"]]
no_alamat = [r for r in records if not r["alamat"]]
print(f"\nMissing fields (sanity):")
print(f"  Tanpa email : {len(no_email)} record(s)")
for r in no_email:
    print(f"    [{r['nomor']}] {r['nama_opd']}")
print(f"  Tanpa telp  : {len(no_telp)} record(s)")
for r in no_telp:
    print(f"    [{r['nomor']}] {r['nama_opd']}")
print(f"  Tanpa alamat: {len(no_alamat)} record(s)")
for r in no_alamat:
    print(f"    [{r['nomor']}] {r['nama_opd']}")
assert len(no_alamat) == 0, "Semua entry harus punya alamat"

# 5. Suspicious: alamat sangat pendek
suspicious = [r for r in records if r["alamat"] and len(r["alamat"]) < 10]
if suspicious:
    print(f"\n⚠ Alamat mencurigakan (terlalu pendek):")
    for r in suspicious:
        print(f"  [{r['nomor']}] {r['nama_opd']}: {r['alamat']!r}")

✓ Total records: 61 (expected 60-65)
✓ Main entries 1-43 lengkap

Distribusi tipe:
  Dinas          : 17
  Kecamatan      : 15
  Bagian         : 9
  Kelurahan      : 9
  Badan          : 4
  Sekretariat    : 2
  RSUD           : 2
  Inspektorat    : 1
  Satpol         : 1
  Kantor         : 1

Missing fields (sanity):
  Tanpa email : 1 record(s)
    [1] Sekretariat Daerah
  Tanpa telp  : 2 record(s)
    [25.f] Kelurahan Proyonanggan Utara
    [36] Kecamatan Pecalungan
  Tanpa alamat: 0 record(s)


## Step 4 — Convert ke LangChain Documents

Format `page_content` agar embedding-friendly & keyword-rich. Metadata Chroma tidak support list, jadi `no_telp` di-join string.

In [6]:
from langchain_core.documents import Document


def record_to_document(rec):
    lines = [
        f"Nama OPD: {rec['nama_opd']}",
    ]
    if rec["parent_opd"]:
        lines.append(f"Bagian dari: {rec['parent_opd']}")
    lines.append(f"Tipe: {rec['tipe']}")
    lines.append(f"Alamat: {rec['alamat']}")
    lines.append(f"Email: {rec['email'] if rec['email'] else '-'}")
    telp_str = ", ".join(rec["no_telp"]) if rec["no_telp"] else "-"
    lines.append(f"No. Telp: {telp_str}")
    content = "\n".join(lines)

    metadata = {
        "source": "Nama dan Alamat OPD Kab Batang.pdf",
        "doc_type": "opd_directory",
        "nomor": rec["nomor"],
        "nama_opd": rec["nama_opd"],
        "parent_opd": rec["parent_opd"] or "",
        "tipe": rec["tipe"],
        "page": rec["page"],
        "has_email": bool(rec["email"]),
        "has_telp": bool(rec["no_telp"]),
    }
    return Document(page_content=content, metadata=metadata)


cleaned_opd_docs = [record_to_document(r) for r in records]
print(f"Total Documents: {len(cleaned_opd_docs)}")
print(f"\n=== Sample Document #0 ===")
print(cleaned_opd_docs[0].page_content)
print(f"\nMetadata: {cleaned_opd_docs[0].metadata}")
print(f"\n=== Sample Document #1 (sub-entry) ===")
print(cleaned_opd_docs[1].page_content)
print(f"\nMetadata: {cleaned_opd_docs[1].metadata}")

Total Documents: 61

=== Sample Document #0 ===
Nama OPD: Sekretariat Daerah
Tipe: Sekretariat
Alamat: Jl. RA Kartini No. 1 Batang
Email: -
No. Telp: (0285) 391571

Metadata: {'source': 'Nama dan Alamat OPD Kab Batang.pdf', 'doc_type': 'opd_directory', 'nomor': '1', 'nama_opd': 'Sekretariat Daerah', 'parent_opd': '', 'tipe': 'Sekretariat', 'page': 1, 'has_email': False, 'has_telp': True}

=== Sample Document #1 (sub-entry) ===
Nama OPD: Bagian Pemerintahan
Bagian dari: Sekretariat Daerah
Tipe: Bagian
Alamat: Jl. RA Kartini No. 1 Batang
Email: bag_pemerintahan@batangkab.go.id
No. Telp: (0285) 392729, (0285) 391571

Metadata: {'source': 'Nama dan Alamat OPD Kab Batang.pdf', 'doc_type': 'opd_directory', 'nomor': '1.a', 'nama_opd': 'Bagian Pemerintahan', 'parent_opd': 'Sekretariat Daerah', 'tipe': 'Bagian', 'page': 1, 'has_email': True, 'has_telp': True}


## Step 5 — Export ke Pickle

In [7]:
import pickle

OUTPUT_PATH = Path("../data/cleaned_opd_docs.pkl")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_PATH, "wb") as f:
    pickle.dump(cleaned_opd_docs, f)

print(f"Saved {len(cleaned_opd_docs)} docs → {OUTPUT_PATH.resolve()}")

Saved 61 docs → C:\Users\Nafisha\Documents\RAGTrial\data\cleaned_opd_docs.pkl


## Step 6 — Manual Quality Verification

Tampilkan semua Documents untuk cross-check vs PDF. Bandingkan side-by-side dengan file PDF asli.

In [8]:
for i, doc in enumerate(cleaned_opd_docs):
    m = doc.metadata
    print(f"[{m['nomor']}] {m['nama_opd']}  ({m['tipe']})")
    if m['parent_opd']:
        print(f"     Parent : {m['parent_opd']}")
    # Re-extract dari content untuk display
    for line in doc.page_content.split("\n")[1:]:
        if line.startswith("Bagian dari:") or line.startswith("Tipe:") or line.startswith("Nama OPD:"):
            continue
        print(f"     {line}")
    print("─" * 70)

[1] Sekretariat Daerah  (Sekretariat)
     Alamat: Jl. RA Kartini No. 1 Batang
     Email: -
     No. Telp: (0285) 391571
──────────────────────────────────────────────────────────────────────
[1.a] Bagian Pemerintahan  (Bagian)
     Parent : Sekretariat Daerah
     Alamat: Jl. RA Kartini No. 1 Batang
     Email: bag_pemerintahan@batangkab.go.id
     No. Telp: (0285) 392729, (0285) 391571
──────────────────────────────────────────────────────────────────────
[1.b] Bagian Kesejahteraan Rakyat  (Bagian)
     Parent : Sekretariat Daerah
     Alamat: Jl. RA Kartini No. 1 Batang
     Email: bag_kesra@batangkab.go.id
     No. Telp: (0285) 391412, (0285) 391571
──────────────────────────────────────────────────────────────────────
[1.c] Bagian Hukum  (Bagian)
     Parent : Sekretariat Daerah
     Alamat: Jl. RA Kartini No. 1 Batang
     Email: bag_hukum@batangkab.go.id
     No. Telp: (0285) 391810, (0285) 391571
──────────────────────────────────────────────────────────────────────
[1.d] Bagi

In [9]:
import statistics

lengths = [len(d.page_content) for d in cleaned_opd_docs]
print(f"=== Summary Statistics ===")
print(f"Total Documents : {len(cleaned_opd_docs)}")
print(f"Length min/mean/max: {min(lengths)} / {statistics.mean(lengths):.0f} / {max(lengths)} chars")
print(f"\nDistribusi tipe:")
for t, n in Counter(d.metadata['tipe'] for d in cleaned_opd_docs).most_common():
    print(f"  {t:15s}: {n}")
print(f"\nMain entries vs sub-entries:")
main_count = sum(1 for d in cleaned_opd_docs if "." not in d.metadata['nomor'])
sub_count = len(cleaned_opd_docs) - main_count
print(f"  Main : {main_count}")
print(f"  Sub  : {sub_count}")
print(f"\nField coverage:")
print(f"  has_email : {sum(1 for d in cleaned_opd_docs if d.metadata['has_email'])}/{len(cleaned_opd_docs)}")
print(f"  has_telp  : {sum(1 for d in cleaned_opd_docs if d.metadata['has_telp'])}/{len(cleaned_opd_docs)}")

=== Summary Statistics ===
Total Documents : 61
Length min/mean/max: 115 / 157 / 208 chars

Distribusi tipe:
  Dinas          : 17
  Kecamatan      : 15
  Bagian         : 9
  Kelurahan      : 9
  Badan          : 4
  Sekretariat    : 2
  RSUD           : 2
  Inspektorat    : 1
  Satpol         : 1
  Kantor         : 1

Main entries vs sub-entries:
  Main : 43
  Sub  : 18

Field coverage:
  has_email : 60/61
  has_telp  : 59/61
